**웹 검색 · 날씨 예보 · 환율 도구를 조합해 여행 준비 정보를 구조화된 형태로 돌려주는 에이전트를 만들어라.**

* 도구

    * TavilySearch: 현지 행사 · 주의사항 등 최신 정보 검색

    * get_forecast(city, days): Open-Meteo로 향후 N일(최대 7일) 최고 · 최저 기온과 강수 확률을 조회한다. 일별 예보는 daily 파라미터를 사용한다.

    * get_exchange_rate(currency): 무료 환율 API(https://open.er-api.com/v6/latest/{통화코드})로 외화 1단위당 원화를 조회한다.

* 두 API 도구 모두 호출 실패와 잘못된 입력을 {"error": ...} 형태로 반환한다.

* response_format에 TripBrief 스키마를 지정한다.

    * 필드: city, weather_summary, packing(준비물 목록), krw_per_unit, budget_tip, local_news, sources(URL 목록)

* stream(stream_mode="updates")로 도구 호출 진행 상황을 출력하면서, 같은 실행에서 나온 structured_response를 받아 JSON으로 출력한다.

In [3]:
import json
import os

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field

load_dotenv()

MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL)

# 현지 행사 · 주의사항 검색 도구
tavily = TavilySearch(max_results=3, topic="news")

In [4]:
@tool
def get_forecast(city: str, days: int = 3) -> dict:
    """도시의 향후 N일(최대 7일) 일별 최고 · 최저 기온과 강수 확률을 조회합니다. city는 영문 도시명."""
    try:
        if not 1 <= days <= 7:
            return {"error": f"days는 1~7 범위여야 함: {days}"}
        if not city.strip():
            return {"error": "city가 비어 있음"}

        # 도시명 -> 좌표 (지오코딩)
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en"},
            timeout=10,
        )
        geo.raise_for_status()
        results = geo.json().get("results")
        if not results:
            return {"error": f"도시를 찾을 수 없음: {city}"}
        place = results[0]

        # 일별 예보 (daily 파라미터)
        res = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                "timezone": "auto",
                "forecast_days": days,
            },
            timeout=10,
        )
        res.raise_for_status()
        daily = res.json()["daily"]
        return {
            "city": place["name"],
            "country": place.get("country"),
            "forecast": [
                {
                    "date": daily["time"][i],
                    "max_c": daily["temperature_2m_max"][i],
                    "min_c": daily["temperature_2m_min"][i],
                    "precip_prob": daily["precipitation_probability_max"][i],
                }
                for i in range(len(daily["time"]))
            ],
        }
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


@tool
def get_exchange_rate(currency: str) -> dict:
    """외화 1단위당 원화(KRW) 환율을 조회합니다. currency는 통화 코드(USD, JPY, EUR 등)."""
    try:
        code = currency.strip().upper()
        if len(code) != 3 or not code.isalpha():
            return {"error": f"통화 코드가 올바르지 않음: {currency}"}

        res = requests.get(f"https://open.er-api.com/v6/latest/{code}", timeout=10)
        res.raise_for_status()
        data = res.json()
        if data.get("result") != "success":
            return {"error": f"환율 조회 실패: {data.get('error-type', 'unknown')}"}
        if "KRW" not in data["rates"]:
            return {"error": "KRW 환율 없음"}
        return {"currency": code, "krw_per_unit": data["rates"]["KRW"]}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

In [5]:
class TripBrief(BaseModel):
    city: str = Field(description="여행 도시")
    weather_summary: str = Field(description="예보 요약 (기온 범위, 강수 확률)")
    packing: list[str] = Field(description="준비물 목록")
    krw_per_unit: float = Field(description="현지 통화 1단위당 원화")
    budget_tip: str = Field(description="환율을 반영한 예산 팁")
    local_news: str = Field(description="현지 행사 · 주의사항 요약")
    sources: list[str] = Field(description="참고한 URL 목록")


trip_agent = create_agent(
    model=llm,
    tools=[tavily, get_forecast, get_exchange_rate],
    response_format=TripBrief,
    system_prompt=(
        "당신은 여행 준비 도우미입니다. "
        "날씨는 get_forecast, 환율은 get_exchange_rate, 현지 행사 · 주의사항은 웹 검색으로 확인하세요. "
        "도구가 error를 반환하면 지어내지 말고 해당 항목에 확인하지 못했다고 적으세요. "
        "도구 결과에 근거해 한국어로 답하고, sources에는 검색 결과의 URL만 넣으세요."
    ),
)

In [6]:
question = {
    "messages": [
        {"role": "user", "content": "다음 주에 도쿄로 3일 여행 가는데 준비 정보를 정리해줘"}
    ]
}

brief = None
for chunk in trip_agent.stream(question, stream_mode="updates"):
    for node, update in chunk.items():
        if not update:
            continue
        # 같은 실행에서 나온 구조화 응답 수집
        if "structured_response" in update:
            brief = update["structured_response"]
        msgs = update.get("messages")
        if not msgs:
            continue
        last = msgs[-1]
        if getattr(last, "tool_calls", None):
            print(f"[{node}] 도구 호출:", [c["name"] for c in last.tool_calls])
        elif node == "tools":
            print(f"[{node}] 결과:", str(last.content)[:100])

print(json.dumps(brief.model_dump(), ensure_ascii=False, indent=2))

[model] 도구 호출: ['get_forecast', 'get_exchange_rate', 'tavily_search']
[tools] 결과: {"currency": "JPY", "krw_per_unit": 8.406764}
[tools] 결과: {"city": "Tokyo", "country": "Japan", "forecast": [{"date": "2026-10-28", "max_c": 17.8, "min_c": 11
[tools] 결과: {"query": "Tokyo events travel advisories next week upcoming events Tokyo June 2025", "follow_up_que
[model] 도구 호출: ['tavily_search']
[tools] 결과: {"query": "Tokyo events November 2-8 2026 official Tokyo event calendar travel advisories November 2
{
  "city": "도쿄",
  "weather_summary": "요청하신 다음 주 예보는 확인하지 못했습니다. 조회된 가장 가까운 3일(10월 28~30일)은 최고 16.9~17.8°C, 최저 9.7~11.5°C이며 강수확률은 10~39%입니다. 여행 직전에 예보를 다시 확인하세요.",
  "packing": [
    "얇은 겉옷과 긴팔 옷(조회된 가까운 예보 기준)",
    "접이식 우산",
    "편한 워킹화",
    "교통카드 또는 비접촉 결제 수단",
    "여권과 여행자보험 정보"
  ],
  "krw_per_unit": 8.406764,
  "budget_tip": "조회 환율은 1엔당 약 8.41원입니다. 대략 1,000엔은 약 8,407원으로 계산하고, 카드·환전 수수료를 고려해 여유 예산을 잡으세요.",
  "local_news": "도쿄 국제 예술·문화 축제 ARTE TOKYO가 2026년 10월 10일~12월 31일 열리며, 공식 안내에는 마루노우

---

**로그인한 고객 정보를 ToolRuntime으로 도구에 전달해, 고객이 자기 주문만 조회 · 환불할 수 있는 상담 에이전트를 만들어라.**

* context_schema: Customer(customer_id, name) dataclass

* 도구

    * list_my_orders(runtime): runtime.context.customer_id의 주문만 반환한다. 모델이 넘기는 인자는 없다.

    * request_refund(order_id, reason, runtime)

* 본인 주문이 아니거나 없는 주문이면 ValueError를 발생시킨다.

* 배송완료가 아니면 환불 불가 메시지를 반환한다.

* 조건을 모두 통과하면 상태를 "환불요청"으로 바꾼다.

* @wrap_tool_call 미들웨어로 도구 예외를 ToolMessage로 변환하고, ModelCallLimitMiddleware(run_limit=6)로 모델 호출 횟수를 제한한다.

* InMemorySaver 체크포인터로 멀티턴 대화를 유지한다. 호출할 때 config(thread_id)와 context(고객 정보)를 함께 전달한다.

* 테스트 시나리오

    * 고객 A: 주문 조회 → "그중 텀블러 환불"(이전 대화 기억) → 배송중 상품 환불 시도 → 고객 B의 주문번호로 환불 시도

    * 고객 B: 주문 조회 (본인 주문만 보이는지 확인)

In [7]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL)

In [8]:
@dataclass
class Customer:
    customer_id: str
    name: str


# 주문 데이터 (환불 요청 시 status가 바뀜)
ORDERS = {
    "O-1001": {"customer_id": "C001", "item": "텀블러", "status": "배송완료"},
    "O-1002": {"customer_id": "C001", "item": "노트북 파우치", "status": "배송중"},
    "O-1003": {"customer_id": "C001", "item": "에코백", "status": "배송완료"},
    "O-2001": {"customer_id": "C002", "item": "무선 키보드", "status": "배송완료"},
    "O-2002": {"customer_id": "C002", "item": "마우스 패드", "status": "배송중"},
}


@tool
def list_my_orders(runtime: ToolRuntime[Customer]) -> list[dict]:
    """로그인한 고객 본인의 주문 목록을 조회합니다."""
    cid = runtime.context.customer_id
    return [
        {"order_id": oid, "item": o["item"], "status": o["status"]}
        for oid, o in ORDERS.items()
        if o["customer_id"] == cid
    ]


@tool
def request_refund(order_id: str, reason: str, runtime: ToolRuntime[Customer]) -> str:
    """본인 주문의 환불을 요청합니다. 배송완료 상태만 환불 가능."""
    order = ORDERS.get(order_id)
    # 없는 주문과 타인 주문을 같은 메시지로 처리 (타인 주문 존재 여부 비노출)
    if order is None or order["customer_id"] != runtime.context.customer_id:
        raise ValueError(f"본인 주문이 아니거나 존재하지 않는 주문: {order_id}")
    if order["status"] != "배송완료":
        return f"환불 불가: {order_id}는 '{order['status']}' 상태임 (배송완료만 환불 가능)"
    order["status"] = "환불요청"
    return f"환불 요청 접수: {order_id} ({order['item']}), 사유: {reason}"


@wrap_tool_call
def handle_tool_errors(request, handler):
    # 도구 예외를 ToolMessage로 변환해 모델이 사용자에게 안내하도록 함
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"도구 오류: {e}",
            tool_call_id=request.tool_call["id"],
        )

In [9]:
support_agent = create_agent(
    model=llm,
    tools=[list_my_orders, request_refund],
    context_schema=Customer,
    checkpointer=InMemorySaver(),
    middleware=[
        handle_tool_errors,
        ModelCallLimitMiddleware(run_limit=6),
    ],
    system_prompt=(
        "당신은 쇼핑몰 상담 에이전트입니다. "
        "주문 조회는 list_my_orders, 환불은 request_refund로 처리하세요. "
        "도구 오류나 환불 불가 메시지는 그대로 근거로 삼아 한국어로 간결하게 안내하세요."
    ),
)


def ask(text, customer, thread_id):
    # config(thread_id)와 context(고객 정보)를 함께 전달
    result = support_agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"configurable": {"thread_id": thread_id}},
        context=customer,
    )
    print(f"[{customer.name}] {text}")
    print("  ->", result["messages"][-1].content, "\n")


customer_a = Customer(customer_id="C001", name="고객 A")
customer_b = Customer(customer_id="C002", name="고객 B")

In [10]:
# 고객 A: 같은 thread_id로 멀티턴 진행
ask("내 주문 목록 보여줘", customer_a, "thread-A")
ask("그중 텀블러 환불해줘. 사유는 단순 변심", customer_a, "thread-A")
ask("노트북 파우치도 환불해줘. 사유는 단순 변심", customer_a, "thread-A")
ask("주문번호 O-2001 환불해줘. 사유는 불량", customer_a, "thread-A")

# 고객 B: 본인 주문만 보이는지 확인
ask("내 주문 목록 보여줘", customer_b, "thread-B")

[고객 A] 내 주문 목록 보여줘
  -> [{'type': 'text', 'text': '주문 목록입니다.\n\n- O-1001 — 텀블러 — 배송완료\n- O-1002 — 노트북 파우치 — 배송중\n- O-1003 — 에코백 — 배송완료', 'annotations': [], 'id': 'msg_0389ec52b7cf9ee3006ae178eccae087d094a1df1e1b92d38b', 'phase': 'final_answer'}] 

[고객 A] 그중 텀블러 환불해줘. 사유는 단순 변심
  -> [{'type': 'text', 'text': '텀블러(O-1001) 환불 요청이 접수됐습니다. 사유는 단순 변심입니다.', 'annotations': [], 'id': 'msg_0389ec52b7cf9ee3006ae178efeef087d0bc273d9f023a122c', 'phase': 'final_answer'}] 

[고객 A] 노트북 파우치도 환불해줘. 사유는 단순 변심
  -> [{'type': 'text', 'text': '노트북 파우치(O-1002)는 현재 배송 중이라 환불할 수 없습니다. 배송 완료 상태인 주문만 환불 가능합니다.', 'annotations': [], 'id': 'msg_0389ec52b7cf9ee3006ae178f33f6087d0bcb503deca909a06', 'phase': 'final_answer'}] 

[고객 A] 주문번호 O-2001 환불해줘. 사유는 불량
  -> [{'type': 'text', 'text': 'O-2001은 본인 주문이 아니거나 존재하지 않는 주문이라 환불을 요청할 수 없습니다.', 'annotations': [], 'id': 'msg_0389ec52b7cf9ee3006ae178f73ccc87d0ad7c3a57bd77a04c', 'phase': 'final_answer'}] 

[고객 B] 내 주문 목록 보여줘
  -> [{'type': 'text', 'text': '주문 목록입니다.\n\n- 주문번호